In [14]:
import json
import pathlib
# reading the 5-fold json file
fold = 4
json_path = '/hdd/yang/projects/glomeruli_segmentation/2025-zhou-hipct-hierarchical-segmentation/data/nnUNet_preprocessed/Dataset010_25-08Glom_search_w_fat_label_partly_filtered/splits_final.json'
with open(json_path, 'r') as f:
    data = json.load(f)
train_files = data[fold]['train']
print(f"Number of training files in fold {fold}: {len(train_files)}")

Number of training files in fold 4: 1073


In [15]:
# calculate Dice coefficient
import numpy as np


def compute_tp_fp_fn_tn(mask_ref, mask_pred, ignore_mask=None):
    if ignore_mask is None:
        use_mask = np.ones_like(mask_ref, dtype=bool)
    else:
        use_mask = ~ignore_mask
    tp = np.sum((mask_ref & mask_pred) & use_mask)
    fp = np.sum(((~mask_ref) & mask_pred) & use_mask)
    fn = np.sum((mask_ref & (~mask_pred)) & use_mask)
    tn = np.sum(((~mask_ref) & (~mask_pred)) & use_mask)
    return tp, fp, fn, tn


def generate_dice_scores(groundtruth, prediction):

    prediction = np.expand_dims(prediction, axis=0)
    prediction = np.expand_dims(prediction, axis=0)
    groundtruth = np.expand_dims(groundtruth, axis=0)
    groundtruth = np.expand_dims(groundtruth, axis=0)

    # nnUNet
    tp, fp, fn, tn = compute_tp_fp_fn_tn(groundtruth, prediction)
    if tp + fp + fn == 0:
        dice_score = np.nan
    else:
        dominator = np.sum(groundtruth) + np.sum(prediction)
        dice_score = 2 * tp / dominator

    return dice_score

In [16]:
import skimage.io as skio
gt_dir = pathlib.Path('/hdd/yang/projects/glomeruli_segmentation/2025-zhou-hipct-hierarchical-segmentation/data/nnUNet_raw/Dataset010_25-08Glom_search_w_fat_label_partly_filtered/labelsTr/')
pred_dir = pathlib.Path(f'/hdd/yang/projects/glomeruli_segmentation/2025-zhou-hipct-hierarchical-segmentation/results/prediction_on_train/Dataset010/fold{fold}/')

dice_scores = []
for train_name in train_files:
    gt_path = gt_dir / f"{train_name}.tif"
    pred_path = pred_dir / f"{train_name}.tif"

    gt_img = skio.imread(str(gt_path))
    pred_img = skio.imread(str(pred_path))
    dice_score = generate_dice_scores(gt_img, pred_img)
    dice_scores.append(dice_score)
print(f"Average Dice score on training set of fold {fold}: {np.nanmean(dice_scores)}")



Average Dice score on training set of fold 4: 0.8160018410938574


In [17]:
scores = [0.8122, 0.8143, 0.8184, 0.8143, 0.8160]
print(f"Average Dice score on validation set across 5 folds: {np.mean(scores)}")
print(f"Standard Deviation of Dice scores on validation set across 5 folds: {np.std(scores)}")

Average Dice score on validation set across 5 folds: 0.81504
Standard Deviation of Dice scores on validation set across 5 folds: 0.0020674622124720802


In [1]:
# merge two predictions
import skimage.io as skio
import numpy as np
import pathlib
pred_voi2 = skio.imread('/hdd/yang/projects/hipct/hipct_registration/results/LADAF-2021-17_kidney-right_voi2/13.0_to_25.0_resample_label_slice_0_4558.tif')
pred_voi3 = skio.imread('/hdd/yang/projects/hipct/hipct_registration/results/LADAF-2021-17_kidney-right_voi3/13.0_to_25.0_resample_label_slice_0_4558.tif')
print('voi2 shape:', pred_voi2.shape)
print('voi3 shape:', pred_voi3.shape)
merged_pred = np.zeros_like(pred_voi2)
merged_pred[(pred_voi2 > 0) | (pred_voi3 > 0)] = 1
# check
print('Number of positive pixels in voi2 prediction:', np.sum(pred_voi2 > 0))
print('Number of positive pixels in voi3 prediction:', np.sum(pred_voi3 > 0))
print('Number of positive pixels in merged prediction:', np.sum(merged_pred > 0))

save_dir = '/hdd/yang/projects/hipct/hipct_registration/results/LADAF-2021-17_kidney-right_voi2_voi3_merged/'
pathlib.Path(save_dir).mkdir(parents=True, exist_ok=True)
skio.imsave('/hdd/yang/projects/hipct/hipct_registration/results/LADAF-2021-17_kidney-right_voi2_voi3_merged/13.0_to_25.0_resample_label_slice_0_4558.tif', merged_pred.astype(np.uint8))

voi2 shape: (4558, 2605, 1824)
voi3 shape: (4558, 2605, 1824)
Number of positive pixels in voi2 prediction: 10113047
Number of positive pixels in voi3 prediction: 30655349
Number of positive pixels in merged prediction: 40768396


/tmp/ipykernel_3309307/1220169048.py:18: UserWarning: /hdd/yang/projects/hipct/hipct_registration/results/LADAF-2021-17_kidney-right_voi2_voi3_merged/13.0_to_25.0_resample_label_slice_0_4558.tif is a low contrast image
  skio.imsave('/hdd/yang/projects/hipct/hipct_registration/results/LADAF-2021-17_kidney-right_voi2_voi3_merged/13.0_to_25.0_resample_label_slice_0_4558.tif', merged_pred.astype(np.uint8))


In [2]:
# applying mask
mask = skio.imread('/hdd/yang/projects/glomeruli_segmentation/data/LADAF-2021-17-right/25.0um_LADAF-2021-17_kidney-right_complete-organ/mask/cortex_mask_25um.tif')
final_pred = np.zeros_like(merged_pred)
final_pred[(mask > 0) & (merged_pred > 0)] = 1
skio.imsave('/hdd/yang/projects/hipct/hipct_registration/results/LADAF-2021-17_kidney-right_voi2_voi3_merged/cortex_masked_13.0_to_25.0_resample_label_slice_0_4558.tif', final_pred.astype(np.uint8))

/tmp/ipykernel_3309307/2793123893.py:5: UserWarning: /hdd/yang/projects/hipct/hipct_registration/results/LADAF-2021-17_kidney-right_voi2_voi3_merged/cortex_masked_13.0_to_25.0_resample_label_slice_0_4558.tif is a low contrast image
  skio.imsave('/hdd/yang/projects/hipct/hipct_registration/results/LADAF-2021-17_kidney-right_voi2_voi3_merged/cortex_masked_13.0_to_25.0_resample_label_slice_0_4558.tif', final_pred.astype(np.uint8))


In [ ]:
# check model parameters
import torch
model_path = '/hdd/yang/projects/glomeruli_segmentation/2025-zhou-hipct-hierarchical-segmentation/results/nnUNet_results/Dataset001_Glomeruli/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_1/checkpoint_final.pth'
model_data = torch.load(model_path, weights_only=False)
state_dict = model_data['network_weights']

odict_keys(['encoder.stages.0.0.convs.0.conv.weight', 'encoder.stages.0.0.convs.0.conv.bias', 'encoder.stages.0.0.convs.0.norm.weight', 'encoder.stages.0.0.convs.0.norm.bias', 'encoder.stages.0.0.convs.0.all_modules.0.weight', 'encoder.stages.0.0.convs.0.all_modules.0.bias', 'encoder.stages.0.0.convs.0.all_modules.1.weight', 'encoder.stages.0.0.convs.0.all_modules.1.bias', 'encoder.stages.0.0.convs.1.conv.weight', 'encoder.stages.0.0.convs.1.conv.bias', 'encoder.stages.0.0.convs.1.norm.weight', 'encoder.stages.0.0.convs.1.norm.bias', 'encoder.stages.0.0.convs.1.all_modules.0.weight', 'encoder.stages.0.0.convs.1.all_modules.0.bias', 'encoder.stages.0.0.convs.1.all_modules.1.weight', 'encoder.stages.0.0.convs.1.all_modules.1.bias', 'encoder.stages.1.0.convs.0.conv.weight', 'encoder.stages.1.0.convs.0.conv.bias', 'encoder.stages.1.0.convs.0.norm.weight', 'encoder.stages.1.0.convs.0.norm.bias', 'encoder.stages.1.0.convs.0.all_modules.0.weight', 'encoder.stages.1.0.convs.0.all_modules.0.bia